# Redescarga selectiva de GeoTIFF NDVI bisemanal

1. **Objetivo del notebook**
- Redescargar sólo GeoTIFF bisemanales faltantes o dañados.

2. **Decisiones metodológicas tomadas en la construcción del notebook**
- Usar el diagnóstico existente y escribir únicamente en la ruta externa canónica.

3. **Archivos generados en el notebook y ubicación de guardado**
- GeoTIFF reparados en TESIS_NDVI_ROOT o D:/GeoTIFF NDVI.

Cada sección Markdown posterior explica brevemente qué hace el código que la sigue.


Este notebook descarga solamente los GeoTIFF marcados como dañados o faltantes por `check_ndvi_tiff_integrity.ipynb`. No vuelve a bajar todas las bisemanas.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.paths import *
ensure_directories()


In [ ]:
from datetime import datetime
from pathlib import Path

import pandas as pd


GEE_PROJECT = "iconic-scholar-460700-u8"
AOI_BBOX = [-70.38, -19.23, -68.90, -17.49]
SCALE = 10
MAX_CLOUD_PCT = 60

EPI_YEAR = 2024
EPI_WEEK_1_START = pd.Timestamp("2023-12-31")
WEEKS_PER_PERIOD = 2

OUT_DIR = require_ndvi_external_dir()
PROBLEMS_CSV = DATA_DIAGNOSTICS / "ndvi_pending_tiff_files_to_redownload.csv"
BACKUP_DIR = OUT_DIR / "damaged_backup"

STAT_BANDS = {
    "min": "ndvi_min",
    "std": "ndvi_std",
}
STAT_FOLDERS = {
    "min": "minimo_bisemanal",
    "std": "desvest_bisemanal",
}

# Fallback manual por si todavia no existe el CSV de integridad.
FALLBACK_TASKS = pd.DataFrame([
    {"folder": "minimo_bisemanal", "suffix": "min", "biweek": 3},
    {"folder": "desvest_bisemanal", "suffix": "std", "biweek": 3},
])

print("Configuracion lista.")

In [ ]:
def load_redownload_tasks() -> pd.DataFrame:
    if PROBLEMS_CSV.exists():
        tasks = pd.read_csv(PROBLEMS_CSV)
        tasks = tasks[tasks["status"].isin(["MISSING", "DUPLICATE_FILE", "DAMAGED", "SUSPICIOUS_EMPTY"])].copy()
        tasks = tasks[["folder", "suffix", "biweek"]].drop_duplicates()
    else:
        tasks = FALLBACK_TASKS.copy()

    tasks["biweek"] = tasks["biweek"].astype(int)
    tasks = tasks[tasks["suffix"].isin(STAT_BANDS)]
    return tasks.sort_values(["biweek", "suffix"]).reset_index(drop=True)


tasks = load_redownload_tasks()
tasks

In [ ]:
print("Importando Earth Engine y geemap...")
import ee
import geemap

try:
    ee.Initialize(project=GEE_PROJECT)
    print("Earth Engine initialized.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print("Earth Engine authenticated and initialized.")

In [ ]:
aoi = ee.Geometry.Rectangle(AOI_BBOX)

SCL_BAD_CLASSES = [3, 8, 9, 10, 11]


def mask_s2_scl(image):
    scl = image.select("SCL")
    mask = scl.neq(SCL_BAD_CLASSES[0])
    for cls in SCL_BAD_CLASSES[1:]:
        mask = mask.And(scl.neq(cls))
    return image.updateMask(mask)


def add_ndvi(image):
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("ndvi")
    return image.addBands(ndvi)


selected_biweeks = sorted(tasks["biweek"].unique().tolist())
first_start = EPI_WEEK_1_START + pd.to_timedelta((min(selected_biweeks) - 1) * WEEKS_PER_PERIOD * 7, unit="D")
last_start = EPI_WEEK_1_START + pd.to_timedelta((max(selected_biweeks) - 1) * WEEKS_PER_PERIOD * 7, unit="D")
last_end = last_start + pd.Timedelta(days=(WEEKS_PER_PERIOD * 7) - 1)

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(aoi)
      .filterDate(first_start.strftime("%Y-%m-%d"), (last_end + pd.Timedelta(days=1)).strftime("%Y-%m-%d"))
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD_PCT))
      .map(mask_s2_scl)
      .map(add_ndvi))

ndvi_collection = s2.select("ndvi")
print(f"Bisemanas a redescargar: {selected_biweeks}")
print(f"Rango Sentinel-2 consultado: {first_start.date()} a {last_end.date()}")
print(f"Escenas despues del filtro de nubes: {s2.size().getInfo()}")

In [ ]:
def period_image_for_biweek(biweek: int):
    period_start = EPI_WEEK_1_START + pd.to_timedelta((biweek - 1) * WEEKS_PER_PERIOD * 7, unit="D")
    period_end = period_start + pd.Timedelta(days=(WEEKS_PER_PERIOD * 7) - 1)
    start = ee.Date(period_start.strftime("%Y-%m-%d"))
    end = ee.Date((period_end + pd.Timedelta(days=1)).strftime("%Y-%m-%d"))

    return (ndvi_collection
            .filterDate(start, end)
            .reduce(ee.Reducer.mean()
                    .combine(ee.Reducer.min(), sharedInputs=True)
                    .combine(ee.Reducer.max(), sharedInputs=True)
                    .combine(ee.Reducer.stdDev(), sharedInputs=True))
            .rename(["ndvi_mean", "ndvi_min", "ndvi_max", "ndvi_std"])
            .clip(aoi)
            .set({
                "epi_year": EPI_YEAR,
                "period": biweek,
                "period_start": period_start.strftime("%Y-%m-%d"),
                "period_end": period_end.strftime("%Y-%m-%d"),
            }))


period_images = {biweek: period_image_for_biweek(biweek) for biweek in selected_biweeks}

In [ ]:

def output_path_for_task(row) -> Path:
    stat_folder = STAT_FOLDERS[row.suffix]
    return OUT_DIR / stat_folder / f"ndvi_arica_y_parinacota_{EPI_YEAR}_BI{int(row.biweek):02d}_{row.suffix}.tif"


def backup_existing_file(path: Path) -> None:
    if not path.exists():
        return
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_folder = BACKUP_DIR / path.parent.name
    backup_folder.mkdir(parents=True, exist_ok=True)
    backup_path = backup_folder / f"{path.stem}.damaged_{stamp}{path.suffix}"
    path.replace(backup_path)
    print(f"  backup {path.name} -> {backup_path}")


for row in tasks.itertuples(index=False):
    out_path = output_path_for_task(row)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    backup_existing_file(out_path)

    print(f"download {out_path}")
    geemap.download_ee_image(
        image=period_images[int(row.biweek)].select(STAT_BANDS[row.suffix]),
        filename=str(out_path),
        region=aoi,
        scale=SCALE,
        crs="EPSG:32719",
        num_threads=1,
        max_tile_dim=1024,
        max_tile_size=16,
    )

print("Done.")

Después de descargar, vuelve a ejecutar `check_ndvi_tiff_integrity.ipynb` para confirmar que los nuevos TIFF se leen correctamente.

In [ ]:
print("Path migration check completed.")
